In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier

In [24]:
count = 0
for i in range(len(data[72])):
    if(data[72].values[i]==5):
        print(i, '---->', data[72].values[i])
        count += 1
        if count == 5:
            break
# np.count_nonzero(data[72].values==9)

39395 ----> 5.0
93337 ----> 5.0
132567 ----> 5.0
157084 ----> 5.0
188534 ----> 5.0


In [17]:
def load_data(mode='train'):
    path = "data/20191008/david_20191008_session"
    f_ext = ".mat"
    if mode=='train':
        mode_path = "1_"
        data = pd.DataFrame()
        for i in range(1, 11):
            value = str(i).zfill(2)
            load_path = path + mode_path + value + f_ext
            file_data = loadmat(load_path)
            mdata = file_data['data']
            temp = pd.DataFrame(mdata)
            data = pd.concat([data, temp])
        return data
    elif mode=='test':
        mode_path = "2_"
        data = pd.DataFrame()
        for i in range(1, 11):
            value = str(i).zfill(2)
            load_path = path + mode_path + value + f_ext
            file_data = loadmat(load_path)
            mdata = file_data['data']
            temp = pd.DataFrame(mdata)
            data = pd.concat([data, temp])
        return data

In [35]:
def band_pass_filter(eeg, freq_range):
    info = mne.create_info(64, 512, ch_types=["eeg"] * 64)
    raw = mne.io.RawArray(eeg.T, info)
    raw.filter(freq_range[0], freq_range[1], fir_design='firwin')

    return raw._data.T

In [51]:
def drop_classes(df):
    label_not_inc = list(range(2,9))
    indexes_to_drop = []
    i = 0
    while i < len(df):
        if df[72].values[i] in label_not_inc:
            list2 = list(range(i, 2048+i))
            i += 2048
            indexes_to_drop.extend(list2)
        else:
            i += 1
    indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
    df_sliced = df.take(list(indexes_to_keep))

    df_sliced = df_sliced.reset_index(drop=True)
    return df_sliced

In [97]:
def data_win(sfreq, data, asynch_label):
    sampling_window = 2 * sfreq
    shift_length = 1 * sfreq
    t_start = 0

    new_data = []
    labels = []

    while t_start + sampling_window < data.shape[0]:
        new_data.append(data[t_start:t_start+sampling_window, :].T)
        labels.append(asynch_label[t_start:t_start+sampling_window])

        t_start = t_start + shift_length

    return np.array(new_data), np.array(labels)

def transform_label(label_new):
    label = []
    for i in label_new:
        count1 = np.count_nonzero(i==1)
        count9 = np.count_nonzero(i==2)
        if count1 >= 512:
            to_add = 1
        elif count9 >= 512:
            to_add = 2
        else:
            to_add = 0
        label.append(to_add)
        
    label = np.array(label)
    return label

In [53]:
def prune_records(data_new, label):
    train_data = []
    label_data = []

    for i in range(len(label)):
        if label[i] != 0:
            label_data.append(label[i])
            train_data.append(data_new[i])
    label_data = np.array(label_data)
    train_data = np.array(train_data)

    return train_data, label_data

In [54]:
def fbcsp(df, labels, sfreq, train=True, csp_objects=None):
    freq = 4
    increment = 4
    end_freq = 40
    if train==True:
        csp_objects = []
        csp_data = []
        while freq < end_freq:
            freq_range = []
            freq_range.append(freq)
            freq_range.append(freq+increment)
            freq += increment
            out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

            X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
            y = transform_label(y)
            X, y = prune_records(X, y)

            csp = CSP(n_components=2, reg=None, log=True, norm_trace=False)
            final_data = csp.fit_transform(X, y)

            csp_objects.append(csp)
            csp_data.append(final_data)

        return np.array(csp_objects), np.array(csp_data), y

    else:
        csp_data = []
        count = 0
        while freq < end_freq:
            freq_range = []
            freq_range.append(freq)
            freq_range.append(freq+increment)
            freq += increment
            out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

            X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
            y = transform_label(y)
            X, y = prune_records(X, y)

            final_data = csp_objects[count].transform(X)
            count += 1

            csp_data.append(final_data)

        return np.array(csp_data), y

In [60]:
def convert_class(sfreq, trigger_points, df):
    asynch_label = []
    i = 0
    while i < len(df[72]):
        if(df[72].values[i] == 1):
            end = (sfreq * trigger_points[1]) + 1
            for j in range(i, i+end):
                asynch_label.append(1)
            i += end
        elif(df[72].values[i] == 9):
            end = (sfreq * trigger_points[9]) + 1
            for j in range(i, i+end):
                asynch_label.append(2)
            i += end
        elif(df[72].values[i] == 0):
            asynch_label.append(0)
            i += 1

    return np.array(asynch_label)

In [108]:
def itr(n_class, p_class, c_time):
    B = (np.log2(n_class) + (p_class * np.log2(p_class)) + ((1-p_class) * np.log2((1-p_class)/(n_class-1)))) / c_time * 60

    return B

def performance_metrics(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    print('Accuracy Score: ', acc)
    print('Cohen Kappa Score: ', cohen_kappa_score(y_test, y_pred))
    print('ITR (bits per minute): ', itr(n_class=2, p_class=acc, c_time=2))
    print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))

In [99]:
def get_train_data():
    sfreq = 512
    trigger_points = {1:4, 9:4}
    
    df = load_data(mode='train')
    train_df = drop_classes(df)
    
    train_df = train_df.drop([64, 65, 66, 67, 68, 69, 70, 71], axis=1)
    asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)
    csp_objects, csp_data, y = fbcsp(train_df, asynch_label, sfreq, train=True)
    
    final_data = pd.DataFrame(csp_data[0])
    col_count = 4
    for i in range(1, len(csp_data)):
        for j in range(len(csp_data[i].T)):
            final_data[str(col_count)] = csp_data[i].T[j]
            col_count += 1

    return final_data.values, y, csp_objects

def get_eval_data(csp_objects):
    sfreq = 512
    trigger_points = {1:4, 9:4}
    
    df = load_data(mode='test')
    train_df = drop_classes(df)
    train_df = train_df.drop([64, 65, 66, 67, 68, 69, 70, 71], axis=1)
    
    asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)
    csp_objects, csp_data, y = fbcsp(train_df, asynch_label, sfreq, train=True)
    
    final_data = pd.DataFrame(csp_data[0])
    col_count = 4
    for i in range(1, len(csp_data)):
        for j in range(len(csp_data[i].T)):
            final_data[str(col_count)] = csp_data[i].T[j]
            col_count += 1

    return final_data.values, y

In [33]:
def train_mibif(X, y):
    print('------Train---------')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

  #get the best k features base on MIBIF algorithm
    select_K = SelectKBest(mutual_info_classif,k=10).fit(X, y)
    extra = select_K.get_support()
    if extra[0] == True:
        extra[1] = True
    for i in range(2, len(extra)):
        if extra[i] == True:
            if i%2 == 0:
                extra[i+1] = True
            else:
                extra[i-1] = True
    pos = np.where(extra==False)
    New_train = np.delete(X_train, list(pos[0]), 1)
    New_test = np.delete(X_test, list(pos[0]), 1)
    # New_train=select_K.transform(X_train)
    # New_test=select_K.transform(X_test)
    ss = StandardScaler()
    New_train = ss.fit_transform(New_train,y_train)
    New_test = ss.transform(New_test)

    print('####### SVM#####')
    svm = SVC()
    svm.fit(New_train, y_train)
    y_pred = svm.predict(New_test)
    performance_metrics(y_test, y_pred)

    print('##########LDA#########')
    lda = LinearDiscriminantAnalysis()
    lda.fit(New_train, y_train)
    y_pred = lda.predict(New_test)
    performance_metrics(y_test, y_pred)

    return list(pos[0]), svm, lda, ss

def eval_mibif(pos, svm, lda, ss, X, y):
    print('------Test--------')
    X = np.delete(X, pos, 1)
    X = ss.transform(X)
    print('#####SVM######')
    y_pred = svm.predict(X)
    performance_metrics(y, y_pred)
    print('#####LDA######')
    y_pred = lda.predict(X)
    performance_metrics(y, y_pred)

In [34]:
def train_rf(X, y):
    print('--------Train--------')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    clf = RandomForestClassifier(random_state=42)
    ss = StandardScaler()
    X_train = ss.fit_transform(X_train, y_train)
    clf.fit(X_train, y_train)
    X_test = ss.transform(X_test)
    y_pred = clf.predict(X_test)
    ####Random Forest#######
    performance_metrics(y_test, y_pred)

    return clf, ss
def eval_rf(clf, ss, X, y):
    print('--------Test------')
    X = ss.transform(X)
    y_pred = clf.predict(X)
    performance_metrics(y, y_pred)

In [100]:
X, y, csp_objects = get_train_data()
X_eval, y_eval = get_eval_data(csp_objects=csp_objects)

Creating RawArray with float64 data, n_channels=64, n_times=1288704
    Range : 0 ... 1288703 =      0.000 ...  2516.998 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 845 samples (1.650 sec)

Computing data rank from raw with rank=None
    Using tolerance 78 (2.2e-16 eps * 64 dim * 5.5e+15  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Computing data r

- Lower transition bandwidth: 7.00 Hz (-6 dB cutoff frequency: 24.50 Hz)
- Upper passband edge: 32.00 Hz
- Upper transition bandwidth: 8.00 Hz (-6 dB cutoff frequency: 36.00 Hz)
- Filter length: 241 samples (0.471 sec)

Computing data rank from raw with rank=None
    Using tolerance 1e-10 (2.2e-16 eps * 64 dim * 7.1e+03  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Computing data rank from raw with rank=None
    Using tolerance 1.2e-10 (2.2e-16 eps * 64 dim * 8.7e+03  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=64, n_times=1288704
    Range : 0 ... 1288703 =      0.000 ...  2516.998 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up b

Done.
Computing data rank from raw with rank=None
    Using tolerance 0.0015 (2.2e-16 eps * 64 dim * 1e+11  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=64, n_times=1121280
    Range : 0 ... 1121279 =      0.000 ...  2189.998 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 20 - 24 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 20.00
- Lower transition bandwidth: 5.00 Hz (-6 dB cutoff frequency: 17.50 Hz)
- Upper passband edge: 24.00 Hz
- Upper transition bandwidth: 6.00 Hz (-6 dB cutoff frequency: 27.00 Hz)
- Filter length: 339 samples (0.662 sec)

Co

In [107]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[40  0]
 [ 0 40]]
##########LDA#########
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[40  0]
 [ 0 40]]
------Test--------
#####SVM######
Accuracy Score:  0.8340248962655602
Cohen Kappa Score:  0.668226872246696
ITR (bits per minute):  0.002929879428801665
Confusion Matrix:  [[108  12]
 [ 28  93]]
#####LDA######
Accuracy Score:  0.7053941908713693
Cohen Kappa Score:  0.4107173606088783
ITR (bits per minute):  0.0010450259132675141
Confusion Matrix:  [[83 37]
 [34 87]]


C:\Users\Admin\anaconda3\envs\mne\lib\site-packages\ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
C:\Users\Admin\anaconda3\envs\mne\lib\site-packages\ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  
C:\Users\Admin\anaconda3\envs\mne\lib\site-packages\ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
C:\Users\Admin\anaconda3\envs\mne\lib\site-packages\ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  


In [105]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  1.0
Cohen Kappa Score:  1.0
ITR (bits per minute):  nan
Confusion Matrix:  [[40  0]
 [ 0 40]]
--------Test------
Accuracy Score:  0.8423236514522822
Cohen Kappa Score:  0.6846418732782369
ITR (bits per minute):  11.1384535271402
Confusion Matrix:  [[101  19]
 [ 19 102]]


C:\Users\Admin\anaconda3\envs\mne\lib\site-packages\ipykernel_launcher.py:2: RuntimeWarning: divide by zero encountered in log2
  
C:\Users\Admin\anaconda3\envs\mne\lib\site-packages\ipykernel_launcher.py:2: RuntimeWarning: invalid value encountered in double_scalars
  
